In [1]:
import os
import json
import subprocess
import shutil
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
ELEVENLABS_API_KEY = "sk_a5ba8a37b338c304ad00df23c7dbb876f86be2e69c0f19ec"
VOICE_ID           = "JBFqnCBsd6RMkjVDRZzb"
MODEL_ID           = "eleven_turbo_v2_5"

ORIGINAL_VIDEO     = "video.mp4"
AUDIO_DIR          = "audio_clips"
TEMP_DIR           = "temp_clips"
OUTPUT_VIDEO       = "final_demo.mp4"
AUDIO_META_PATH    = os.path.join(AUDIO_DIR, "audio_meta.json")
WORD_TIMES_PATH    = os.path.join(AUDIO_DIR, "word_timestamps.json")

# Caption style
FONT_NAME          = "Arial"
FONT_SIZE          = 28
NORMAL_COLOR       = "&H00FFFFFF"   # white
HIGHLIGHT_COLOR    = "&H0000FFFF"   # yellow
OUTLINE_COLOR      = "&H00000000"   # black outline
BACK_COLOR         = "&H80000000"   # semi-transparent black bg
MARGIN_V           = 40             # px from bottom

os.makedirs(TEMP_DIR, exist_ok=True)

# ─────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────
def run(cmd, label=""):
    print(f"▶ {label}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"❌ Error:\n{result.stderr[-600:]}")
        raise RuntimeError(label)
    return result

def get_duration(path):
    r = subprocess.run([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        path
    ], capture_output=True, text=True)
    try:
        return float(r.stdout.strip())
    except:
        return 0.0

def seconds_to_ass(s):
    """Convert seconds to ASS time format H:MM:SS.cc"""
    cs = int((s % 1) * 100)   # centiseconds
    ss = int(s) % 60
    m  = int(s // 60) % 60
    h  = int(s // 3600)
    return f"{h}:{m:02d}:{ss:02d}.{cs:02d}"

# ─────────────────────────────────────────
# STEP 1: GET WORD-LEVEL TIMESTAMPS FROM ELEVENLABS
# Uses the /with-timestamps endpoint
# Returns character-level timing → we convert to word timing
# ─────────────────────────────────────────
def get_word_timestamps(text, narration_start_s):
    """
    Call ElevenLabs TTS with timestamps.
    Returns list of: {word, start_s, end_s}
    where start_s/end_s are absolute times in the final video.
    """
    url = f"https://api.elevenlabs.io/v1/text-to-speech/{VOICE_ID}/with-timestamps"

    headers = {
        "xi-api-key":   ELEVENLABS_API_KEY,
        "Content-Type": "application/json"
    }

    payload = {
        "text":     text,
        "model_id": MODEL_ID,
        "voice_settings": {
            "stability":         0.5,
            "similarity_boost":  0.75,
            "style":             0.3,
            "use_speaker_boost": True
        }
    }

    response = requests.post(url, headers=headers, json=payload, timeout=30)

    if response.status_code != 200:
        print(f"  ❌ ElevenLabs error {response.status_code}: {response.text[:200]}")
        return [], None

    data        = response.json()
    audio_b64   = data.get("audio_base64", "")
    alignment   = data.get("alignment", {})

    chars       = alignment.get("characters", [])
    char_starts = alignment.get("character_start_times_seconds", [])
    char_ends   = alignment.get("character_end_times_seconds", [])

    if not chars:
        return [], audio_b64

    # ── Convert character timestamps → word timestamps ──
    words       = []
    cur_word    = ""
    word_start  = None

    for i, (ch, cs, ce) in enumerate(zip(chars, char_starts, char_ends)):
        if ch == " " or i == len(chars) - 1:
            # end of word
            if ch != " ":
                cur_word += ch
                ce_final  = ce
            else:
                ce_final = char_ends[i-1] if i > 0 else ce

            if cur_word.strip():
                words.append({
                    "word":    cur_word.strip(),
                    "start_s": round(narration_start_s + word_start, 4),
                    "end_s":   round(narration_start_s + ce_final,   4),
                })
            cur_word   = ""
            word_start = None
        else:
            if word_start is None:
                word_start = cs
            cur_word += ch

    return words, audio_b64

# ─────────────────────────────────────────
# STEP 2: LOAD AUDIO META + REGENERATE WITH TIMESTAMPS
# We need to re-call ElevenLabs with /with-timestamps
# to get word timing (existing audio files don't have this)
# ─────────────────────────────────────────
print("🔧 Loading audio metadata...")

with open(AUDIO_META_PATH) as f:
    audio_meta = json.load(f)

video_duration = get_duration(ORIGINAL_VIDEO)
print(f"📹 Original video: {round(video_duration, 2)}s")

# Calculate sequential timestamps first
valid_clips     = []
total_narration = 0.0

for entry in audio_meta:
    audio_path = os.path.join(AUDIO_DIR, entry["audio"])
    if not os.path.exists(audio_path):
        continue
    dur = get_duration(audio_path)
    valid_clips.append((entry, audio_path, dur))
    total_narration += dur

# Assign cumulative timestamps
audio_entries   = []
cumulative_time = 0.0

for entry, audio_path, dur in valid_clips:
    audio_entries.append({
        "audio_path":  audio_path,
        "timestamp_s": round(cumulative_time, 4),
        "end_s":       round(cumulative_time + dur, 4),
        "duration":    dur,
        "narration":   entry.get("narration", ""),
        "position":    entry.get("position", "middle"),
    })
    cumulative_time += dur

master_duration = cumulative_time
print(f"⏱️  Master duration: {round(master_duration, 2)}s")

# ─────────────────────────────────────────
# STEP 3: FETCH WORD TIMESTAMPS IN PARALLEL
# ─────────────────────────────────────────
# Check if already cached
if os.path.exists(WORD_TIMES_PATH):
    print(f"\n⚡ Loading cached word timestamps...")
    with open(WORD_TIMES_PATH) as f:
        all_word_timestamps = json.load(f)
else:
    print(f"\n⚡ Fetching word timestamps from ElevenLabs ({len(audio_entries)} clips)...")

    all_word_timestamps = []

    def fetch_words(args):
        idx, ae = args
        print(f"  [{idx+1}/{len(audio_entries)}] Fetching: {ae['narration'][:50]}...")
        words, _ = get_word_timestamps(ae["narration"], ae["timestamp_s"])
        return idx, words

    results = [None] * len(audio_entries)

    with ThreadPoolExecutor(max_workers=3) as ex:
        futures = {
            ex.submit(fetch_words, (i, ae)): i
            for i, ae in enumerate(audio_entries)
        }
        for future in as_completed(futures):
            idx, words = future.result()
            results[idx] = words

    for words in results:
        if words:
            all_word_timestamps.extend(words)

    # Cache for re-runs
    with open(WORD_TIMES_PATH, "w") as f:
        json.dump(all_word_timestamps, f, indent=2)

print(f"✅ Total words with timestamps: {len(all_word_timestamps)}")

# ─────────────────────────────────────────
# STEP 4: GENERATE KARAOKE ASS FILE
#
# ASS karaoke works by putting ALL words of a line
# in one dialogue block, then using {\k} tags
# to highlight each word in sequence.
#
# Format: {\k<duration_cs>}word  (duration in centiseconds)
# ─────────────────────────────────────────
print(f"\n🎨 Generating karaoke ASS subtitles...")

ASS_HEADER = f"""[Script Info]
ScriptType: v4.00+
PlayResX: 1920
PlayResY: 1080
Collisions: Normal

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Karaoke,{FONT_NAME},{FONT_SIZE},{NORMAL_COLOR},{HIGHLIGHT_COLOR},{OUTLINE_COLOR},{BACK_COLOR},1,0,0,0,100,100,0,0,3,2,0,2,20,20,{MARGIN_V},1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""

# Group words into lines (by narration block)
# Each narration = one ASS dialogue line
ass_events = []

word_idx = 0
for ae in audio_entries:
    narration_words = ae["narration"].split()
    line_words      = []

    # Collect word timestamps for this narration block
    for word in narration_words:
        if word_idx < len(all_word_timestamps):
            line_words.append(all_word_timestamps[word_idx])
            word_idx += 1

    if not line_words:
        continue

    line_start = line_words[0]["start_s"]
    line_end   = line_words[-1]["end_s"]

    # Build karaoke text: {\kN}word for each word
    # N = duration of this word in centiseconds
    karaoke_text = ""
    for w in line_words:
        dur_cs       = int((w["end_s"] - w["start_s"]) * 100)
        dur_cs       = max(dur_cs, 5)   # minimum 5cs per word
        karaoke_text += f"{{\\k{dur_cs}}}{w['word']} "

    karaoke_text = karaoke_text.strip()

    # Wrap into multiple ASS lines if sentence is too long (>10 words)
    if len(line_words) > 10:
        # Split into chunks of 8 words
        chunk_size = 8
        chunks     = [line_words[i:i+chunk_size]
                      for i in range(0, len(line_words), chunk_size)]

        for chunk in chunks:
            chunk_start = chunk[0]["start_s"]
            chunk_end   = chunk[-1]["end_s"]
            chunk_text  = ""
            for w in chunk:
                dur_cs      = max(int((w["end_s"] - w["start_s"]) * 100), 5)
                chunk_text += f"{{\\k{dur_cs}}}{w['word']} "

            ass_events.append(
                f"Dialogue: 0,{seconds_to_ass(chunk_start)},"
                f"{seconds_to_ass(chunk_end)},Karaoke,,0,0,0,,"
                f"{chunk_text.strip()}"
            )
    else:
        ass_events.append(
            f"Dialogue: 0,{seconds_to_ass(line_start)},"
            f"{seconds_to_ass(line_end)},Karaoke,,0,0,0,,"
            f"{karaoke_text}"
        )

ass_content = ASS_HEADER + "\n".join(ass_events)
ass_path    = os.path.join(TEMP_DIR, "karaoke.ass")

with open(ass_path, "w", encoding="utf-8") as f:
    f.write(ass_content)

print(f"✅ ASS file: {len(ass_events)} caption blocks")

# ─────────────────────────────────────────
# STEP 5: STRETCH VIDEO + BUILD AUDIO IN PARALLEL
# ─────────────────────────────────────────
stretched_video = os.path.join(TEMP_DIR, "stretched.mp4")
narration_track = os.path.join(TEMP_DIR, "narration.aac")
slow_factor     = master_duration / video_duration if master_duration > video_duration else 1.0
need_stretch    = master_duration > video_duration

def stretch_video():
    if need_stretch:
        print(f"\n🎞️  [PARALLEL] Stretching video {round(slow_factor,2)}x...")
        run([
            "ffmpeg", "-y",
            "-i", ORIGINAL_VIDEO,
            "-vf", f"setpts={slow_factor}*PTS",
            "-an", "-c:v", "libx264",
            "-preset", "fast", "-crf", "18",
            stretched_video
        ], label="Stretching video")
        return stretched_video
    return ORIGINAL_VIDEO

def build_narration():
    print(f"\n🎚️  [PARALLEL] Building narration audio...")
    inputs        = ["-f", "lavfi", "-i", "anullsrc=r=44100:cl=stereo"]
    filter_parts  = [f"[0:a]atrim=duration={master_duration},asetpts=PTS-STARTPTS[base]"]
    stream_labels = ["[base]"]

    for idx, ae in enumerate(audio_entries):
        delay_ms = int(ae["timestamp_s"] * 1000)
        inputs  += ["-i", ae["audio_path"]]
        filter_parts.append(
            f"[{idx+1}:a]adelay={delay_ms}|{delay_ms},"
            f"apad=whole_dur={master_duration}[d{idx}]"
        )
        stream_labels.append(f"[d{idx}]")

    all_labels = "".join(stream_labels)
    filter_parts.append(
        f"{all_labels}amix=inputs={len(stream_labels)}:"
        f"duration=first:normalize=0[aout]"
    )

    run(
        ["ffmpeg", "-y"] + inputs + [
            "-filter_complex", ";".join(filter_parts),
            "-map", "[aout]",
            "-c:a", "aac", "-b:a", "192k", "-ar", "44100",
            "-t", str(master_duration),
            narration_track
        ],
        label="Building narration"
    )
    return narration_track

print(f"\n⚡ Stretching video + building audio in parallel...")
with ThreadPoolExecutor(max_workers=2) as ex:
    f_video = ex.submit(stretch_video)
    f_audio = ex.submit(build_narration)
    source_video   = f_video.result()
    narration_path = f_audio.result()

print(f"✅ Video: {round(get_duration(source_video),2)}s")
print(f"✅ Audio: {round(get_duration(narration_path),2)}s")

# ─────────────────────────────────────────
# STEP 6: FINAL RENDER WITH KARAOKE CAPTIONS
# ─────────────────────────────────────────
print(f"\n🎬 Final render with karaoke captions...")

source_video_abs = Path(source_video).resolve().as_posix()
narration_abs    = Path(narration_path).resolve().as_posix()
ass_abs          = Path(ass_path).resolve().as_posix()
output_abs       = Path(OUTPUT_VIDEO).resolve()
original_dir     = os.getcwd()

# Copy ASS to TEMP_DIR for relative path (Windows fix)
shutil.copy(ass_path, os.path.join(TEMP_DIR, "karaoke.ass"))

os.chdir(TEMP_DIR)
try:
    run([
        "ffmpeg", "-y",
        "-i", source_video_abs,
        "-i", narration_abs,
        "-map", "0:v",
        "-map", "1:a",
        "-vf", "ass=karaoke.ass",       # ✅ relative path — no Windows colon bug
        "-c:v", "libx264",
        "-preset", "fast",
        "-crf", "18",
        "-c:a", "aac", "-b:a", "192k",
        "-t", str(master_duration),
        "-movflags", "+faststart",
        str(output_abs)
    ], label="Rendering with karaoke captions")
finally:
    os.chdir(original_dir)

shutil.rmtree(TEMP_DIR)
print("🧹 Cleaned up")

size_mb   = os.path.getsize(OUTPUT_VIDEO) / (1024 * 1024)
final_dur = get_duration(OUTPUT_VIDEO)

print(f"\n{'─'*60}")
print(f"✅ FINAL VIDEO READY")
print(f"{'─'*60}")
print(f"Output:          {OUTPUT_VIDEO}")
print(f"Duration:        {round(final_dur, 2)}s ({round(final_dur/60, 2)} min)")
print(f"Caption style:   Karaoke — highlighted word ✅")
print(f"Position:        Bottom center ✅")
print(f"Words tracked:   {len(all_word_timestamps)}")
print(f"Size:            {round(size_mb, 2)} MB")
print(f"{'─'*60}")

🔧 Loading audio metadata...
📹 Original video: 88.86s
⏱️  Master duration: 178.56s

⚡ Fetching word timestamps from ElevenLabs (17 clips)...
  [1/17] Fetching: Welcome to VisionCurator, the platform designed to...
  [2/17] Fetching: On the Datasets page, you can see an overview of y...
  [3/17] Fetching: Let's explore our 'dogs' dataset. VisionCurator au...
  [4/17] Fetching: You can easily browse through all images, gaining ...
  [5/17] Fetching: VisionCurator intelligently organizes your images ...
  [6/17] Fetching: Here, we see images grouped into distinct clusters...
  [7/17] Fetching: This clustering capability is crucial for identify...
  [8/17] Fetching: Beyond clusters, VisionCurator also identifies out...
  [9/17] Fetching: These outliers represent unique or unusual example...
  [10/17] Fetching: Now, let's leverage this intelligence with VisionC...
  [11/17] Fetching: Smart Export downloads a perfectly balanced datase...
  [12/17] Fetching: Simply specify your desired target 

SameFileError: 'temp_clips\\karaoke.ass' and 'temp_clips\\karaoke.ass' are the same file